# CICIoT2023 exploratory analysis

This notebook inspects the processed splits produced by `python -m src.data`.
It writes tables to `results/eda_*.csv` and figures to `figures/eda_*.png`.
Run it only after the dataset has been downloaded and split.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import load_config
from src.data import FAMILY_NAMES, MULTICLASS_LABELS
from src.features import FEATURE_NAMES

config = load_config("configs/default.yaml")
processed_dir = Path(config["data"]["processed_dir"])
results_dir = Path(config["paths"]["results"])
figures_dir = Path(config["paths"]["figures"])
results_dir.mkdir(exist_ok=True)
figures_dir.mkdir(exist_ok=True)

In [ ]:
splits = {
    name: pd.read_pickle(processed_dir / f"{name}.pkl")
    for name in ("train", "val", "test")
}
train, val, test = splits["train"], splits["val"], splits["test"]

split_summary = pd.DataFrame(
    {
        name: {
            "rows": len(frame),
            "classes": frame["label_multiclass"].nunique(),
            "families": frame["label_family"].nunique(),
            "capture_files": frame["source_file"].nunique(),
        }
        for name, frame in splits.items()
    }
).T
split_summary.to_csv(results_dir / "eda_split_summary.csv")
split_summary

In [ ]:
label_counts = (
    train.groupby(["label_family", "label_multiclass"], observed=True)
    .size()
    .rename("count")
    .reset_index()
)
family_counts = (
    train["label_family"]
    .value_counts()
    .reindex(FAMILY_NAMES, fill_value=0)
    .rename_axis("family")
    .reset_index(name="count")
)
label_counts.to_csv(results_dir / "eda_label_counts_train.csv", index=False)
family_counts.to_csv(results_dir / "eda_family_counts_train.csv", index=False)
family_counts

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4))
axis.bar(family_counts["family"], family_counts["count"])
axis.set_ylabel("Rows")
axis.set_title("Training rows by attack family")
axis.tick_params(axis="x", rotation=30)
figure.tight_layout()
figure.savefig(figures_dir / "eda_family_counts.png", dpi=200)
plt.close(figure)

In [ ]:
distribution = (
    pd.concat(
        [frame.assign(split=name) for name, frame in splits.items()],
        ignore_index=True,
    )
    .groupby(["split", "label_multiclass"], observed=True)
    .size()
    .rename("count")
    .reset_index()
)
pivot = (
    distribution.pivot(
        index="label_multiclass",
        columns="split",
        values="count",
    )
    .reindex(MULTICLASS_LABELS)
    .fillna(0)
)
pivot.to_csv(results_dir / "eda_class_distribution.csv")

figure, axis = plt.subplots(figsize=(16, 5))
pivot.plot(kind="bar", ax=axis, width=0.8)
axis.set_ylabel("Rows")
axis.set_title("Class distribution by split")
axis.tick_params(axis="x", labelsize=6, rotation=90)
figure.tight_layout()
figure.savefig(figures_dir / "eda_class_distribution.png", dpi=200)
plt.close(figure)
pivot.head()

In [ ]:
feature_columns = [name for name in FEATURE_NAMES if name in train.columns]
raw_features = train[feature_columns]
cleaned_features = raw_features.replace([np.inf, -np.inf], np.nan)
missingness = pd.DataFrame(
    {
        "missing_fraction": cleaned_features.isna().mean(),
        "infinite_fraction": raw_features.isin([np.inf, -np.inf]).mean(),
    }
).sort_values("missing_fraction", ascending=False)
missingness.to_csv(results_dir / "eda_missingness.csv")
missingness.head(15)

In [ ]:
feature_summary = cleaned_features.describe().T
feature_summary.to_csv(results_dir / "eda_feature_summary.csv")
feature_summary.head()

In [ ]:
sample = cleaned_features.dropna(axis=1, how="all").sample(
    n=min(50_000, len(train)),
    random_state=int(config["seed"]),
)
correlation = sample.corr()

figure, axis = plt.subplots(figsize=(12, 10))
image = axis.imshow(correlation, cmap="coolwarm", vmin=-1.0, vmax=1.0)
axis.set_xticks(range(len(correlation.columns)))
axis.set_yticks(range(len(correlation.index)))
axis.set_xticklabels(correlation.columns, rotation=90, fontsize=6)
axis.set_yticklabels(correlation.index, fontsize=6)
axis.set_title("Feature correlation (training sample)")
figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
figure.tight_layout()
figure.savefig(figures_dir / "eda_feature_correlation.png", dpi=200)
plt.close(figure)

## Notes

- Tables are written under `results/eda_*.csv`; figures under `figures/eda_*.png`.
- Review class imbalance, missingness, and capture-level split leakage before
  interpreting the baseline or attack metrics.
- Treat high correlations as expected: several CICIoT2023 features are aggregates
  of the same packet-size and timing statistics.